# TAB 62: exp62_relenergy_polarity (exp60 base + relative_energy_anchor + patch_polarity_align) - Colab
# H14: Decoupled spatial energy contrast (FM11 fix) + patch-consistent polarity alignment (TMR fix with broadband Dodd-Deeds phase preservation)

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phải thấy từ Drive (lỗi cũ: train dùng data_dir='data' local -> 0 scans)
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

TDMS: 81
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [3]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git pull --ff-only && git log --oneline -3

remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 17 (delta 12), reused 17 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 141.13 KiB | 1.99 MiB/s, done.
From https://github.com/TranAnh35/GraduationProject
   f436e30..e99f6de  Define_PECT_JEPA -> origin/Define_PECT_JEPA
Updating f436e30..e99f6de
Fast-forward
 src/PECT_JEPA/cross_condition/colab/train_60.ipynb | 594 ++++++++++++++++++++-
 src/PECT_JEPA/cross_condition/colab/train_61.ipynb | 379 ++++++++++++-
 src/PECT_JEPA/cross_condition/colab/train_62.ipynb | 294 ++++++++++
 src/PECT_JEPA/cross_condition/configs/config.py    |   2 +
 .../cross_condition/knowledge/graph.sqlite         | Bin 2748416 -> 3158016 bytes
 .../cross_condition/knowledge/seed_causal.json     |  17 +-
 src/PECT_JEPA/cross_condition/models/tokenizer.py  |  37 +-
 src/PECT_JEPA/cross_condition/train.py             |   4 +
 8 files changed, 1267 inser

In [5]:
# [FIX] sklearn < version co sample_weight trong MLPClassifier.fit -> phai EP DUNG 1.7.2 (giong env lab).
# pip install tran se bi bo qua vi Colab da co san sklearn cu.
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"


torch 2.11.0+cu130 | cuda: True | sklearn: 1.7.2


In [ ]:
!mkdir -p "$DRIVE_RUNS"
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo LINKED; else echo ALREADY_LINKED; fi
!cd GraduationProject && ls -la experiments/cross_condition/ | grep runs

ALREADY_LINKED
lrwxrwxrwx 1 root root   41 Oct 18 14:58 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# ===== CHỌN 1 EXP CHO TAB NÀY =====
EXP_NAME = 'exp62_relenergy_polarity'
BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --inv_start_epoch 11 --inv_ramp_epochs 20 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --inv_cosine true --zc_cond_weight 0.1 --var_gamma 0.5 --offset_weight 0.0 --pool_var_weight 0.0 --device cuda'
EXTRA_ALL = {
    'exp60_contpos_asyminv_fullctx': '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05',
    'exp62_relenergy_polarity': '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --patch_polarity_align true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05',
}
EXTRA = EXTRA_ALL[EXP_NAME]
os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print(EXP_NAME)
print(EXTRA)

exp62_relenergy_polarity
--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --patch_polarity_align true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05


In [7]:
# Chuyen runs sang Drive KHONG SUA CODE: symlink runs -> DRIVE_RUNS (chay 1 lan sau cell config).
# Tu day moi run/checkpoint/metrics ghi thang len Drive; cell backup chi con de phong.
!mkdir -p "$DRIVE_RUNS"
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


lrwxrwxrwx 1 root root   41 Oct 18 16:33 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [ ]:
# Chunk 1 (fresh): CHỈ chạy cell này khi bắt đầu run mới (KHÔNG chạy cùng lúc với cell resume01 ở tab khác).
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

[16:33:32 | INFO] Initialized Unified Experiment Directory: experiments/cross_condition/runs/exp62_relenergy_polarity_20261018_163332
[16:33:32 | INFO] Configuration: CrossConditionConfig(exp_name='exp62_relenergy_polarity', add_timestamp=True, base_exp_dir='experiments/cross_condition/runs', embed_dim=64, encoder_depth=4, encoder_heads=4, predictor_depth=2, predictor_heads=4, mlp_ratio=4.0, dropout=0.0, spatial_topology='concentric_star', star_radii=(2, 4, 7), grid_size=5, temporal_samples=128, num_harmonics=16, num_target_probes=9, num_context_probes=16, continuous_pos_embed=True, contiguous_masking=True, predictor_scattering_only=False, scan_centered_inv=True, patch_energy_norm=False, median_patch_energy=False, clean_phasor_token=False, subspace_norm=False, relative_energy_anchor=True, patch_polarity_align=True, use_condition_token=True, condition_modulation='query_projection', tokenizer_type='energy_adaptive', relative_target=False, latent_disentanglement=True, carrier_dim=32, scat

In [ ]:
# ==============================================================================
# Resume training tiep tuc tu DRIVE_RUNS (CHỈ chạy khi session trước đã dừng hẳn)
# ==============================================================================
import os, glob

# 1. Tim run folder trong DRIVE_RUNS (uu tien run moi nhat hoac chi dinh run cu the)
SPECIFIC_RUN = ""  # De trong de tu dong lay run moi nhat, hoac dien vi du: "exp60_contpos_asyminv_fullctx_20261017_223000"

if SPECIFIC_RUN:
    run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN)
else:
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

print(f"[*] Thu muc run tren Drive: {run_dir}")

# 2. Xac dinh checkpoint (uu tien latest de tiep tuc epoch & optimizer/scaler state, fallback sang best)
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")

if os.path.exists(ckpt_latest):
    resume_ckpt = ckpt_latest
elif os.path.exists(ckpt_best):
    resume_ckpt = ckpt_best
else:
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint resume: {resume_ckpt} ({os.path.getsize(resume_ckpt) / 1024**2:.1f} MB)")

# 3. Kiem tra tien do da train tu metrics_epoch.csv
csv_path = os.path.join(run_dir, "evaluation_results", "metrics_epoch.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(run_dir, "metrics_epoch.csv")
if os.path.exists(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        if len(lines) > 1:
            last_line = lines[-1].split(",")
            print(f"[*] Da hoan thanh epoch: {last_line[0]} | Val AUC gan nhat: {last_line[14] if len(last_line) > 14 else 'N/A'}")

# 4. Dam bao symlink GraduationProject/experiments/cross_condition/runs -> DRIVE_RUNS
os.system(f'mkdir -p GraduationProject/experiments/cross_condition && ln -sfn "{DRIVE_RUNS}" GraduationProject/experiments/cross_condition/runs')

# 5. Export checkpoint path sang environ va thuc hien resume training
os.environ["RESUME_CKPT"] = resume_ckpt
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name "$EXP_NAME" $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

In [ ]:
# ==============================================================================
# Full Downstream Evaluation + Readouts Diagnostic v6 sau khi train xong
# ==============================================================================
import os, glob

# 1. Tim run can evaluate trong DRIVE_RUNS
SPECIFIC_RUN = ""  # De trong de tu dong lay run moi nhat
if SPECIFIC_RUN:
    run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN)
else:
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")

eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest
if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Dang evaluate checkpoint: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda

[*] Dang evaluate checkpoint: /content/drive/MyDrive/PECT_Dataset/runs/exp62_relenergy_polarity_20261018_163332/checkpoints/best_cross_condition_jepa.pt

PECT-JEPA Unified Downstream Benchmark Evaluation
Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp62_relenergy_polarity_20261018_163332/checkpoints/best_cross_condition_jepa.pt
Output Directory: /content/drive/MyDrive/PECT_Dataset/runs/exp62_relenergy_polarity_20261018_163332/evaluation_results
Compute Device:   cuda
Target Splits:    ['train', 'val', 'test']

Discovered 81 total inspection files:
  - Train Split (Base Domain):     12 files
  - Val Split (Base Domain):       4 files
  - Test Split (Held-out OOD):     65 files

STEP 1: Extracting Train Pool Latents & Fitting Global Readout Probes


  [Train Pool Aggregation]
  - Defect Detection samples: 46900 (Defects: 9060, Sound: 37840)
  - Flaw Sizing samples:      6475 defect pixels
  - Latent regime reference (median scan E||z||^2): 1.9
  Fitting global Logistic Reg